<img src="https://raw.githubusercontent.com/Mierkoles/CribbNotes/main/CribbNotesLogo.png" alt="CribbNotes" width="220">

# Mine the Room — Watch a Model Derive Itself From *Your* Data

Welcome — your first hands-on moment.

## First: what is this thing?

If you've never opened a notebook before, two minutes of orientation:

**What you're looking at is a *Jupyter notebook*, opened in *Google Colab*.** A notebook is a document that mixes blocks of writing (like this one) with blocks of code. The code blocks have a little ▶ button — clicking it runs that code on Google's computer and shows the result right below. Think of it as a lab notebook for software: code, results, and notes interleaved on one page.

**Google Colab** is Google's free service for running notebooks in the browser. You don't install anything. You don't need to know Python. Close the tab when you're done; nothing to clean up.

**What you're about to do:** Mark is going to mine the room for data — ten pairs of numbers. We drop them into the notebook and then watch the machine *derive* the two numbers that capture the relationship hidden in those points. Then we'll see what more data does, and what *bad* data does.

**You do not need to read or understand the code.** Every block is commented for the curious, but the demo is in the animations.

> **No GPU needed for this one.** Default Colab runtime (CPU) is fine — save your GPU allocation for the nanoGPT demo later in the session.

## What's in this notebook

- **Setup** — imports and two helper functions
- **Your data** — the ten (x, y) pairs from the room (Mark fills these in live)
- **Part 1 — Derive the model** — start from a random guess and watch *gradient descent* search for the two numbers that fit your points
- **Part 2 — Why more data helps** — treat that relationship as real, then pile on data and watch the estimate sharpen
- **Part 3 — Why bad data hurts** — inject a few wild points and watch two algorithms react
- **Recap** — how this scales to GPT-4


In [ ]:
# -------------------------------------------------------------------
# Setup — imports, a random-number generator, and two helpers
# -------------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rc
from matplotlib.animation import FuncAnimation
from sklearn.linear_model import LinearRegression, HuberRegressor

# Render animations as inline HTML5 players (Colab's standard pattern)
rc("animation", html="jshtml")
rc("animation", embed_limit=50)  # MB cap on embedded animation size

# One seeded RNG so every laptop in the room sees identical randomness
# (otherwise the narration wouldn't match what's on your screen).
RNG = np.random.default_rng(42)

NOISE_STD = 8.0   # used later when we simulate "more data" around the relationship


def fit_ols(x, y):
    """Ordinary least squares: the m and b that minimize summed SQUARED error.
    Closed-form and instant — we use it as the reference 'right answer'
    that Part 1's gradient descent will hunt for the slow way."""
    model = LinearRegression().fit(np.asarray(x).reshape(-1, 1), y)
    return float(model.coef_[0]), float(model.intercept_)


def fit_huber(x, y):
    """Huber regression: a 'robust' fit that treats big errors LINEARLY
    instead of squaring them, so a few wild points can't drag it far."""
    model = HuberRegressor().fit(np.asarray(x).reshape(-1, 1), y)
    return float(model.coef_[0]), float(model.intercept_)


print("Setup loaded. Next cell holds the room's data.")


In [ ]:
# -------------------------------------------------------------------
# YOUR DATA — the ten pairs the room shouts out
#
# Mark: replace the list below with the numbers people call out.
# Each entry is (x, y), two integers from 0 to 100.
# Leave it as-is to run with the backup demo set.
# -------------------------------------------------------------------

# Backup demo set (a loose upward trend) — also the fallback if the
# live data turns out unusable.
HOUSE_POINTS = [(8, 18), (14, 15), (23, 31), (31, 28), (39, 45),
                (48, 42), (57, 60), (66, 58), (81, 77), (93, 88)]

# ⬇️⬇️⬇️  LIVE DATA GOES HERE  ⬇️⬇️⬇️
RAW_POINTS = HOUSE_POINTS
# ⬆️⬆️⬆️  (paste the room's pairs, e.g. [(0,100),(100,0),(50,20), ...])  ⬆️⬆️⬆️


def clean_points(pts):
    """Keep only valid pairs: two numbers, each clamped to 0..100, rounded
    to integers. Anything malformed or out of range is dropped — so no one
    can crash the demo or shove the plot off-screen."""
    out = []
    for p in pts:
        try:
            x, y = float(p[0]), float(p[1])
        except (TypeError, ValueError, IndexError):
            continue
        x = min(100.0, max(0.0, x))
        y = min(100.0, max(0.0, y))
        out.append((round(x), round(y)))
    return out


pts = clean_points(RAW_POINTS)

# --- Guardrails ---------------------------------------------------
# Two ways the live data can be unusable:
#   1. Fewer than 3 valid points — not enough to make a point.
#   2. Everyone picked the SAME x — zero variation in the input, so there's
#      no relationship to find. (Worth saying out loud: a model can't learn
#      from a column of constants — your data has to actually VARY along the
#      thing you're predicting from.)
xs_all = np.array([p[0] for p in pts], dtype=float) if pts else np.array([])
using_fallback = False
if len(pts) < 3 or xs_all.std() < 1e-9:
    print("⚠️  Crowd data unusable (too few points, or no spread in x).")
    print("    Falling back to the backup demo set — but say the lesson out loud:")
    print("    a model needs VARIATION in its inputs to learn anything at all.\n")
    pts = clean_points(HOUSE_POINTS)
    using_fallback = True

x_crowd = np.array([p[0] for p in pts], dtype=float)
y_crowd = np.array([p[1] for p in pts], dtype=float)

# Reference answer (closed-form). Part 1 will rediscover this the slow way.
m_fit, b_fit = fit_ols(x_crowd, y_crowd)

print(f"Using {len(pts)} points" + (" (backup set)" if using_fallback else " from the room") + ".")
print("Part 1 will now search these points for the pattern from scratch — two numbers, derived. Run it.")


## Part 1 — Where do those two numbers come from? We *derive* them.

A model is a function with parameters. Here the parameters are just two numbers, and **the machine has to find them from your data** — we don't get to plant them.

**First, watch your data shape the answer.** We'll drop your points in one at a time. Each time a point lands (highlighted in gold), the computer recomputes its best guess and the red guide jumps to match. With two points it can pass right through them; as more arrive it can't please everyone, so it settles into the best compromise.

> **Hit ▶ when Mark says go.**


In [ ]:
# -------------------------------------------------------------------
# Part 1a — add YOUR points one at a time; the best guess re-settles
# after each. We use the closed-form best fit at every step, so you see
# the DATA pull the guess into shape. (HOW the computer searches for it
# is the next cell.)
# -------------------------------------------------------------------

ns_add = list(range(2, len(x_crowd) + 1))      # reveal points: 2, 3, ... all of them
ns_add += [ns_add[-1]] * 3                      # hold on the full set for a beat

fig, ax = plt.subplots(figsize=(11, 7))


def update_add(i):
    n = ns_add[i]
    ax.clear()

    xn, yn = x_crowd[:n], y_crowd[:n]
    m, b = fit_ols(xn, yn)              # best guess on the points so far

    xs = np.array([0, 100])
    ax.plot(xs, m * xs + b, "r-", lw=2.8, zorder=2)            # unlabeled — no verdict yet

    # points already in play, then the just-added one emphasized in gold
    ax.scatter(xn[:-1], yn[:-1], s=80, alpha=0.85, zorder=3, label="your data")
    ax.scatter(xn[-1:], yn[-1:], s=170, color="#F8B922", edgecolors="black",
               linewidths=1.6, zorder=4, label="just added")

    ax.set_title(f"{n} of {len(x_crowd)} points", fontsize=14, fontweight="bold")
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_xlim(0, 100); ax.set_ylim(0, 100)   # fixed axes
    ax.legend(loc="upper left", fontsize=10)
    ax.grid(alpha=0.3)


anim_add = FuncAnimation(fig, update_add, frames=len(ns_add), interval=750, repeat=False)
plt.close(fig)
anim_add


## …but how does it *find* that answer?

It didn't look up a formula — it **searched** for it. It starts from a **random guess** (two numbers out of nowhere), measures how wrong it is, nudges them in the direction that's less wrong, and repeats. That loop is **gradient descent**, and it *is* training.

**What to watch:**
- The thin grey lines are the **errors** — the gap from each of your points to the model's current guess.
- The title shows the **total error falling** with every step.
- The red guess swings out of its random starting position and settles into the best fit it can find.

> **Hit ▶.**


In [ ]:
# -------------------------------------------------------------------
# Part 1 — Gradient descent, deriving m and b on YOUR points
#
# We run gradient descent on a standardized version of x (so a single
# learning rate behaves well regardless of what numbers the room picked),
# then convert the parameters back to ordinary units for display. The
# math you SEE — the errors and the numbers — is all in your
# real (0..100) units.
# -------------------------------------------------------------------

# Standardize x for well-conditioned descent: z = (x - mean) / std
xm, xsd = x_crowd.mean(), x_crowd.std()
z = (x_crowd - xm) / xsd

# Start from a deliberately-wrong random line (seeded → identical on
# every laptop). This is the "random guess" the model begins with.
g0 = np.random.default_rng(7)
mz = float(g0.uniform(-3.0, 3.0))
bz = float(g0.uniform(0.0, 100.0))

LR = 0.08      # learning rate — how big a step we take downhill each time
STEPS = 35     # how many descent steps to animate

# Run the descent and record each step for the animation.
history = []
for step in range(STEPS + 1):
    pred = mz * z + bz                 # current predictions (in y units)
    err = pred - y_crowd               # how wrong we are at each point
    loss = float((err ** 2).mean())    # mean squared error

    # Convert the standardized params back to real-units m and b for display
    m_disp = mz / xsd
    b_disp = bz - mz * xm / xsd
    history.append((step, m_disp, b_disp, loss))

    # Gradient of the loss w.r.t. each parameter, then a step downhill
    grad_mz = 2.0 * np.mean(err * z)
    grad_bz = 2.0 * np.mean(err)
    mz -= LR * grad_mz
    bz -= LR * grad_bz

fig, ax = plt.subplots(figsize=(11, 7))


def update_part1(frame):
    step, m, b, loss = history[frame]
    ax.clear()

    # Errors being minimized: a thin grey segment from each point to the line
    for xi, yi in zip(x_crowd, y_crowd):
        ax.plot([xi, xi], [yi, m * xi + b], color="gray", lw=0.9, alpha=0.55, zorder=1)

    ax.scatter(x_crowd, y_crowd, s=80, alpha=0.85, zorder=3, label="your data")

    xs = np.array([0, 100])
    # Draw the model's current guess with no legend label, and keep the title
    # to just the error number (set below) — the screen shows the search, not
    # a verdict.
    ax.plot(xs, m * xs + b, "r-", lw=2.8, zorder=2)

    if step == 0:
        head = "Step  0:  random guess"
    elif frame == len(history) - 1:
        head = f"Step {step:2d}:  settled"
    else:
        head = f"Step {step:2d}:  searching..."
    ax.set_title(f"{head}      total error ↓ {loss:,.0f}",
                 fontsize=14, fontweight="bold")

    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_xlim(0, 100); ax.set_ylim(0, 100)   # fixed axes — no one can shove it off-screen
    ax.legend(loc="upper left", fontsize=10)
    ax.grid(alpha=0.3)


anim_part1 = FuncAnimation(fig, update_part1, frames=len(history),
                           interval=220, repeat=False)
plt.close(fig)
anim_part1


**What you just watched:**

- The model began with a **random line** — two arbitrary numbers. It had no idea what your data looked like.
- Step by step, it measured its total error and nudged the slope and intercept *downhill*. That loop — guess, measure, adjust — is **gradient descent**, and it is exactly how every model you've ever heard of is trained.
- It landed on the two numbers (**m** and **b**) that make the line least wrong. **Nobody told it those numbers. It derived them from your data.**

> *And notice: it gave you a clean line, an equation, two decimal places — **no matter what numbers you fed it.** If the room handed it nonsense, it fit a confident line through the nonsense. A model never tells you "this is garbage." It always answers. Knowing when the answer is meaningful is **your** job — and that doesn't go away no matter how good these things get.*

---

**For the curious (and a common question):** *Linear regression* is the **model** — the line itself. *Gradient descent* is the **method** that found it. They're different things that happen to meet here: a line is simple enough to have an exact **formula** (no search needed), so we ran the slow downhill search on purpose — because it's the method that **scales**. GPT-4 has no formula; the only way to find its parameters (a rumored ~1.76 trillion of them) is this same loop. Real systems use faster variants of it — **stochastic gradient descent**, **momentum**, **Adam/AdamW** (the default for transformers) — but the loop you just watched, *measure the error and step downhill*, is exactly it.


## Part 2 — Why does more data help?

We only had a handful of points, so how much should we trust that line? To answer that, we need more data — but we can't squeeze more numbers out of the room. So we'll do the next best thing: **pretend your fitted line is the real relationship**, then simulate collecting more and more noisy measurements of it.

Watch the fitted estimate (red) lock onto your line (dotted) as the sample grows: **5 → 10 → 50 → 500 → 10,000 points.**

> **Hit ▶ when Mark says go.**


In [ ]:
# -------------------------------------------------------------------
# Part 2 — "More data" convergence
# Treat the Part 1 fit (m_fit, b_fit) as the TRUE relationship, simulate
# noisy samples of it, and watch the OLS estimate tighten onto it as N
# grows. (Honest framing: more data didn't change the answer — it made
# us CONFIDENT in it.)
# -------------------------------------------------------------------

TRUE_M, TRUE_B = m_fit, b_fit

rng_b = np.random.default_rng(42)   # fresh seed → Part 2 reproducible on its own
N_MAX = 10_000
x_master = rng_b.uniform(0, 100, N_MAX)
y_master = TRUE_M * x_master + TRUE_B + rng_b.normal(0, NOISE_STD, N_MAX)

ns = [5, 10, 50, 500, 10_000]

# y-limits: cover the true line across 0..100 plus a noise margin
yline = TRUE_M * np.array([0, 100]) + TRUE_B
ylo = float(min(yline.min(), 0) - 15)
yhi = float(max(yline.max(), 100) + 15)

# Pre-compute fits + (subsampled) scatter for each frame
frames_b = []
sub_rng = np.random.default_rng(0)
for n in ns:
    x, y = x_master[:n], y_master[:n]
    m, b = fit_ols(x, y)
    if n > 1500:
        idx = sub_rng.choice(n, 1500, replace=False)
        xp, yp = x[idx], y[idx]
    else:
        xp, yp = x, y
    frames_b.append((n, m, b, xp, yp))

fig, ax = plt.subplots(figsize=(11, 7))


def update_part2(frame):
    n, m, b, xp, yp = frames_b[frame]
    ax.clear()

    if n > 1500:
        ax.scatter(xp, yp, s=6, alpha=0.25, label=f"data (n = {n:,}; showing 1,500)")
    else:
        ax.scatter(xp, yp, s=55, alpha=0.75, label=f"data (n = {n:,})")

    xs = np.array([0, 100])
    ax.plot(xs, TRUE_M * xs + TRUE_B, "k:", lw=1.4, alpha=0.7,
            label=f"your line: y = {TRUE_M:.2f}x + {TRUE_B:.2f}")
    ax.plot(xs, m * xs + b, "r-", lw=2.8, label=f"fit on {n:,} pts: y = {m:.2f}x + {b:.2f}")

    ax.set_title(f"N = {n:,}", fontsize=15, fontweight="bold")
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_xlim(0, 100); ax.set_ylim(ylo, yhi)
    ax.legend(loc="upper left", fontsize=10)
    ax.grid(alpha=0.3)


anim_part2 = FuncAnimation(fig, update_part2, frames=len(frames_b),
                           interval=1800, repeat=False)
plt.close(fig)
anim_part2


**What you just watched:**

- With only 5 points the estimate wobbles — small samples are unstable.
- By 50 it has mostly settled; by 10,000 it sits right on your line.
- **More data didn't find a *different* answer — it made us confident in the one we had.** Once the signal is captured, extra samples are confirmation, not new information.

> *"The hard question stops being 'do we have enough data' and becomes 'do we have the **right** data.' That's where most enterprise AI projects live or die."*


## Part 3 — Why does *bad* data hurt?

Real data is messy: sensor glitches, typos, fat-fingered entries. We take a clean sample around your line and inject a few wildly-wrong points, fitting it two ways:

- **OLS** (red, solid) — the default. Minimizes *squared* error, so one big mistake counts enormously. Sensitive to outliers.
- **Huber** (green, dashed) — a robust alternative that refuses to let a few bad points dominate.

**Hit ▶.** Watch the red line tilt; watch the green line hold.


In [ ]:
# -------------------------------------------------------------------
# Part 3 — anomalies tilt OLS; Huber resists
# Clean sample around YOUR line, then inject 1, 2, 3 wild points.
# -------------------------------------------------------------------

clean_rng = np.random.default_rng(7)
x_clean = clean_rng.uniform(0, 100, 80)
y_clean = TRUE_M * x_clean + TRUE_B + clean_rng.normal(0, NOISE_STD, 80)

# Three anomalies, placed far off YOUR line so they're obviously wrong
anom_x = np.array([20.0, 50.0, 80.0])
anom_line = TRUE_M * anom_x + TRUE_B
anom_y = anom_line + np.array([60.0, -60.0, 70.0])

# y-limits covering everything we'll plot
all_y = np.concatenate([y_clean, anom_y, TRUE_M * np.array([0, 100]) + TRUE_B])
ylo2 = float(all_y.min() - 15)
yhi2 = float(all_y.max() + 15)

fig, ax = plt.subplots(figsize=(11, 7))


def update_part3(k):
    xk = np.concatenate([x_clean, anom_x[:k]])
    yk = np.concatenate([y_clean, anom_y[:k]])
    ax.clear()

    xs = np.array([0, 100])
    ax.plot(xs, TRUE_M * xs + TRUE_B, "k:", lw=1.3, alpha=0.6, label="your line (truth)")

    # Clean points — muted blue so the bad points stand out against them
    ax.scatter(x_clean, y_clean, s=38, alpha=0.55, color="#3b6fb0",
               label=f"clean data (n = {len(x_clean)})", zorder=2)

    # Anomalies — big crimson X's with a black edge: unmissable from the back row
    if k >= 1:
        anom_label = "bad point (anomaly)" if k == 1 else f"bad points ({k} anomalies)"
        ax.scatter(anom_x[:k], anom_y[:k], s=420, marker="X", color="crimson",
                   edgecolors="black", linewidths=1.8, zorder=6, label=anom_label)

    m_ols, b_ols = fit_ols(xk, yk)
    ax.plot(xs, m_ols * xs + b_ols, "r-", lw=2.6,
            label=f"OLS: y = {m_ols:.2f}x + {b_ols:.2f}")

    if k >= 1:
        m_h, b_h = fit_huber(xk, yk)
        ax.plot(xs, m_h * xs + b_h, "g--", lw=2.6,
                label=f"Huber (robust): y = {m_h:.2f}x + {b_h:.2f}")

    label = "anomaly" if k == 1 else "anomalies"
    ax.set_title(f"{k} {label} added", fontsize=14, fontweight="bold")
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_xlim(0, 100); ax.set_ylim(ylo2, yhi2)
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(alpha=0.3)


anim_part3 = FuncAnimation(fig, update_part3, frames=len(anom_x) + 1,
                            interval=2500, repeat=False)
plt.close(fig)
anim_part3


**What you should be seeing:**

- **0 anomalies:** OLS sits on your line. No drama.
- **1 anomaly:** one bad point pulls the red OLS line off course. Huber barely moves.
- **2–3 anomalies:** OLS is visibly, meaningfully wrong. Huber holds.

> *"The algorithm decides how your model fails when reality is messy. A vendor's benchmark is the red line on clean data. The question that predicts production survival is: what happens when you hand it YOUR messy data?"*

---

## Where this goes next

Everything you just watched was **two numbers — m and b — derived from data by searching for the values that make the model least wrong.** GPT-4 has a rumored **~1.76 trillion** of those numbers. The procedure is the same: guess, measure the error, nudge every parameter downhill, repeat. The only things that change are the function's shape, the parameter count, and how much compute you throw at it.

Keep that image in your head for the rest of the talk — every architecture we discuss (tokenization, embeddings, attention, transformers, the nanoGPT demo you'll run later) is a more elaborate version of this exact picture.
